# Análise de Performance de Operadores

> **Projeto:** Contact Center Data Lakehouse on AWS  
> **Autor:** Data Engineering Portfolio  
> **Data:** 2026-07  
> **Objetivo:** Analisar individualmente o desempenho dos operadores, identificar padrões de alta performance, absenteísmo e oportunidades de desenvolvimento.

---

## Dimensões de Performance Analisadas

| Dimensão | Indicador | Fonte |
|----------|-----------|-------|
| **Produtividade** | Chamadas atendidas por operador | tb_chamada |
| **Qualidade** | Nota média de avaliação (0–10) | tb_avaliacao_qualidade |
| **Eficiência** | TMA individual | tb_chamada |
| **Disponibilidade** | % dias de presença | tb_jornada_operador |
| **Score Geral** | Índice composto ponderado | Calculado |

---

**Modelo de avaliação:** Score = 0.4 × Produtividade + 0.4 × Qualidade − 0.2 × Absenteísmo (valores normalizados 0–1)

In [ ]:
# ============================================================
# IMPORTS E CARREGAMENTO DE DADOS
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = os.path.join('..', 'data', 'synthetic', 'output')
np.random.seed(42)

# ============================================================
# DADOS SINTÉTICOS
# ============================================================
N_OPS      = 80
N_CHAMADAS = 50_000
N_AVAL     = 15_000
N_JORNADA  = N_OPS * 260  # ~260 dias úteis por operador

CARGOS = ['AGENTE', 'AGENTE SR', 'SUPERVISOR', 'COORDENADOR']

# tb_operador
tb_operador = pd.DataFrame({
    'id_operador':  range(1, N_OPS + 1),
    'nm_operador':  [f'Op. {chr(65 + i%26)}{i//26+1:02d}' for i in range(N_OPS)],
    'ds_cargo':     np.random.choice(CARGOS, N_OPS, p=[0.60, 0.25, 0.10, 0.05]),
    'fl_supervisor': np.random.choice([0, 1], N_OPS, p=[0.88, 0.12]),
    'id_supervisor': np.random.choice([None] + list(range(1, 10)), N_OPS),
    'ds_equipe':    np.random.choice(['EQUIPE_A', 'EQUIPE_B', 'EQUIPE_C', 'EQUIPE_D'], N_OPS),
})

# Perfis de operadores: alta performance tem mais chamadas e notas
perfil_prod = np.random.choice(['alto', 'medio', 'baixo'], N_OPS, p=[0.2, 0.6, 0.2])
prod_base   = {'alto': 700, 'medio': 450, 'baixo': 250}
qual_base   = {'alto': 8.5, 'medio': 7.0, 'baixo': 5.5}

tb_operador['_perfil']    = perfil_prod
tb_operador['_prod_base'] = [prod_base[p] for p in perfil_prod]
tb_operador['_qual_base'] = [qual_base[p] for p in perfil_prod]

# tb_chamada (com id_operador e duração)
op_ids = np.random.choice(range(1, N_OPS + 1), N_CHAMADAS,
                           p=np.array([prod_base[p] for p in perfil_prod], dtype=float) /
                             sum([prod_base[p] for p in perfil_prod]))
dates = pd.date_range('2025-01-01', '2025-12-31', periods=N_CHAMADAS)

tb_chamada = pd.DataFrame({
    'id_chamada':          range(1, N_CHAMADAS + 1),
    'id_operador':         op_ids,
    'dt_inicio':           dates,
    'st_chamada':          np.random.choice(['ATENDIDA', 'ABANDONADA', 'OCUPADA'], N_CHAMADAS, p=[0.72, 0.15, 0.13]),
    'tp_chamada':          np.random.choice(['ENTRADA', 'SAIDA'], N_CHAMADAS, p=[0.65, 0.35]),
    'nr_duracao_segundos': np.clip(np.random.exponential(180, N_CHAMADAS).astype(int) + 10, 10, 1500),
    'ds_fila':             np.random.choice(['FILA_SAC', 'FILA_VENDAS', 'FILA_SUPORTE', 'FILA_RETENCAO'], N_CHAMADAS),
})
tb_chamada['dia_semana'] = tb_chamada['dt_inicio'].dt.dayofweek  # 0=Seg

# tb_avaliacao_qualidade
aval_op_ids = np.random.choice(range(1, N_OPS + 1), N_AVAL)
notas = []
for op_id in aval_op_ids:
    base = tb_operador.loc[tb_operador['id_operador'] == op_id, '_qual_base'].values
    base = base[0] if len(base) > 0 else 7.0
    nota = np.clip(np.random.normal(base, 0.8), 1, 10)
    notas.append(round(nota, 1))

tb_avaliacao = pd.DataFrame({
    'id_avaliacao':  range(1, N_AVAL + 1),
    'id_operador':   aval_op_ids,
    'nr_nota':       notas,
    'ds_criterio':   np.random.choice(['ABORDAGEM', 'RESOLUCAO', 'EMPATIA', 'CLAREZA'], N_AVAL),
    'dt_avaliacao':  pd.date_range('2025-01-01', periods=N_AVAL, freq='30min').strftime('%Y-%m-%d'),
})

# tb_jornada_operador
dias_uteis = pd.bdate_range('2025-01-01', '2025-12-31')
jornada_rows = []
for op_id in range(1, N_OPS + 1):
    perf = tb_operador.loc[tb_operador['id_operador'] == op_id, '_perfil'].values[0]
    aus_prob = {'alto': 0.03, 'medio': 0.07, 'baixo': 0.14}[perf]
    for dia in dias_uteis:
        presente = np.random.choice([1, 0], p=[1 - aus_prob, aus_prob])
        jornada_rows.append({
            'id_operador':  op_id,
            'dt_jornada':   dia.strftime('%Y-%m-%d'),
            'fl_presente':  presente,
            'dia_semana':   dia.dayofweek,
            'nr_horas':     np.random.uniform(6, 9) if presente else 0,
        })

tb_jornada = pd.DataFrame(jornada_rows)

print('Tabelas carregadas:')
print(f'  tb_operador:   {len(tb_operador):,} operadores')
print(f'  tb_chamada:    {len(tb_chamada):,} chamadas')
print(f'  tb_avaliacao:  {len(tb_avaliacao):,} avaliações')
print(f'  tb_jornada:    {len(tb_jornada):,} registros de jornada')

## 1. Ranking de Produtividade

Identificamos os operadores com maior volume de chamadas atendidas. Esta métrica, isolada, não representa performance completa — deve ser combinada com qualidade.

In [ ]:
# ============================================================
# 1. RANKING DE PRODUTIVIDADE
# ============================================================
chamadas_atend = tb_chamada[tb_chamada['st_chamada'] == 'ATENDIDA']
prod_op = chamadas_atend.groupby('id_operador').size().reset_index(name='chamadas_atendidas')
prod_op = prod_op.merge(tb_operador[['id_operador', 'nm_operador', 'ds_cargo', 'ds_equipe']], on='id_operador', how='left')
prod_op = prod_op.sort_values('chamadas_atendidas', ascending=False)

top15 = prod_op.head(15)
bottom10 = prod_op.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Top 15 ---
cargo_colors = {'AGENTE': '#3498db', 'AGENTE SR': '#2ecc71', 'SUPERVISOR': '#e67e22', 'COORDENADOR': '#9b59b6'}
colors_top = [cargo_colors.get(c, '#95a5a6') for c in top15['ds_cargo']]
bars = axes[0].barh(top15['nm_operador'][::-1], top15['chamadas_atendidas'][::-1],
                    color=colors_top[::-1], edgecolor='white')
axes[0].set_title('Top 15 Operadores — Chamadas Atendidas', fontweight='bold')
axes[0].set_xlabel('Nº de Chamadas Atendidas')

# Adicionar valor e cargo
for bar, (_, row) in zip(bars, top15[::-1].iterrows()):
    axes[0].text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
                 f'{int(bar.get_width()):,}', va='center', fontsize=8)

# Legenda de cargos
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in cargo_colors.items()]
axes[0].legend(handles=legend_patches, title='Cargo', fontsize=9, loc='lower right')

# --- Distribuição geral ---
axes[1].hist(prod_op['chamadas_atendidas'], bins=20,
             color='#3498db', edgecolor='white', alpha=0.8)
media_prod = prod_op['chamadas_atendidas'].mean()
mediana_prod = prod_op['chamadas_atendidas'].median()
axes[1].axvline(media_prod,   color='red',    linestyle='--', linewidth=2, label=f'Média: {media_prod:.0f}')
axes[1].axvline(mediana_prod, color='orange', linestyle='--', linewidth=2, label=f'Mediana: {mediana_prod:.0f}')
axes[1].set_title('Distribuição de Produtividade — Todos os Operadores', fontweight='bold')
axes[1].set_xlabel('Chamadas Atendidas')
axes[1].set_ylabel('Nº de Operadores')
axes[1].legend()

plt.suptitle('Análise de Produtividade dos Operadores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Total de operadores com chamadas: {len(prod_op)}')
print(f'Média de chamadas por operador: {media_prod:.0f}')
print(f'Top operador: {top15.iloc[0]["nm_operador"]} — {top15.iloc[0]["chamadas_atendidas"]:,} chamadas')

print('\nProdutividade por cargo (média):')
print(prod_op.groupby('ds_cargo')['chamadas_atendidas'].mean().round(0).to_string())

## 2. Qualidade do Atendimento

A nota média das avaliações de qualidade reflete a satisfação com o atendimento prestado por cada operador. O scatter plot correlaciona produtividade × qualidade para identificar o quadrante ideal.

In [ ]:
# ============================================================
# 2. QUALIDADE DO ATENDIMENTO
# ============================================================
qual_op = tb_avaliacao.groupby('id_operador')['nr_nota'].agg(['mean', 'std', 'count']).reset_index()
qual_op.columns = ['id_operador', 'nota_media', 'nota_std', 'n_aval']

# JOIN com produtividade
df_perf = prod_op.merge(qual_op, on='id_operador', how='inner')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Scatter produtividade × qualidade ---
cargo_colors = {'AGENTE': '#3498db', 'AGENTE SR': '#2ecc71', 'SUPERVISOR': '#e67e22', 'COORDENADOR': '#9b59b6'}
colors_sc = [cargo_colors.get(c, '#95a5a6') for c in df_perf['ds_cargo']]

scatter = axes[0].scatter(df_perf['chamadas_atendidas'], df_perf['nota_media'],
                           c=colors_sc, s=100, alpha=0.7, edgecolors='white', linewidth=0.5)

# Linhas de média (quadrantes)
med_prod = df_perf['chamadas_atendidas'].median()
med_qual = df_perf['nota_media'].median()
axes[0].axvline(med_prod, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[0].axhline(med_qual, color='gray', linestyle='--', alpha=0.5, linewidth=1)

# Labels dos quadrantes
xlim = axes[0].get_xlim()
ylim = axes[0].get_ylim()
axes[0].text(xlim[0] + (med_prod - xlim[0])*0.5, ylim[0] + (ylim[1]-ylim[0])*0.92,
             'Qualificado', ha='center', color='blue', fontsize=9, style='italic')
axes[0].text(xlim[0] + (xlim[1]-xlim[0])*0.75, ylim[0] + (ylim[1]-ylim[0])*0.92,
             'Estrela ⭐', ha='center', color='green', fontsize=9, fontweight='bold')
axes[0].text(xlim[0] + (med_prod - xlim[0])*0.5, ylim[0] + (ylim[1]-ylim[0])*0.05,
             'Desenvolvimento', ha='center', color='red', fontsize=9, style='italic')
axes[0].text(xlim[0] + (xlim[1]-xlim[0])*0.75, ylim[0] + (ylim[1]-ylim[0])*0.05,
             'Eficiente', ha='center', color='orange', fontsize=9, style='italic')

axes[0].set_title('Produtividade × Qualidade por Operador', fontweight='bold')
axes[0].set_xlabel('Chamadas Atendidas (Produtividade)')
axes[0].set_ylabel('Nota Média de Qualidade (0–10)')

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in cargo_colors.items()]
axes[0].legend(handles=legend_patches, title='Cargo', fontsize=9)

# --- Bar chart das notas médias (top 15 por qualidade) ---
top15_qual = df_perf.nlargest(15, 'nota_media')
top15_qual = top15_qual.sort_values('nota_media', ascending=True)
bar_colors_q = ['#e74c3c' if n < 7 else '#f39c12' if n < 8.5 else '#2ecc71' for n in top15_qual['nota_media']]
bars = axes[1].barh(top15_qual['nm_operador'], top15_qual['nota_media'],
                    color=bar_colors_q, edgecolor='white')
axes[1].axvline(7,   color='orange', linestyle='--', alpha=0.7, label='Nota 7 (Bom)')
axes[1].axvline(8.5, color='green',  linestyle='--', alpha=0.7, label='Nota 8.5 (Excelente)')
axes[1].set_title('Top 15 Operadores por Nota Média de Qualidade', fontweight='bold')
axes[1].set_xlabel('Nota Média (0–10)')
axes[1].set_xlim(0, 11)
axes[1].legend(fontsize=9)
for bar, (_, row) in zip(bars, top15_qual.iterrows()):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{row["nota_media"]:.1f} (n={int(row["n_aval"])})',
                 va='center', fontsize=8)

plt.suptitle('Análise de Qualidade de Atendimento', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Nota média geral: {df_perf["nota_media"].mean():.2f}')
print(f'Correlação produtividade × qualidade: {df_perf["chamadas_atendidas"].corr(df_perf["nota_media"]):.3f}')

# Quadrantes
estrelas = df_perf[(df_perf['chamadas_atendidas'] >= med_prod) & (df_perf['nota_media'] >= med_qual)]
dev      = df_perf[(df_perf['chamadas_atendidas'] <  med_prod) & (df_perf['nota_media'] <  med_qual)]
print(f'\nEstrelas (alta prod + alta qual): {len(estrelas)} operadores ({len(estrelas)/len(df_perf)*100:.0f}%)')
print(f'Desenvolvimento (baixa prod + baixa qual): {len(dev)} operadores ({len(dev)/len(df_perf)*100:.0f}%)')

## 3. TMA por Operador

O TMA individual revela diferenças na velocidade de resolução. Operadores muito rápidos podem estar resolvendo de forma superficial; muito lentos podem precisar de treinamento ou ferramentas melhores.

In [ ]:
# ============================================================
# 3. TMA POR OPERADOR
# ============================================================
df_at = tb_chamada[(tb_chamada['st_chamada'] == 'ATENDIDA') &
                    (tb_chamada['nr_duracao_segundos'] > 0)].copy()

tma_op = df_at.groupby('id_operador').agg(
    tma_medio=('nr_duracao_segundos', 'mean'),
    tma_median=('nr_duracao_segundos', 'median'),
    n_chamadas=('id_chamada', 'count')
).reset_index()
tma_op = tma_op.merge(tb_operador[['id_operador', 'nm_operador', 'ds_cargo']], on='id_operador', how='left')

# Top 10 com maior TMA (potencial de melhoria)
top10_tma = tma_op.nlargest(10, 'tma_medio').sort_values('tma_medio', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Boxplot por operador (top 10 em TMA) ---
op_order = top10_tma['id_operador'].tolist()
op_labels = {row['id_operador']: row['nm_operador'] for _, row in top10_tma.iterrows()}

data_box = [df_at[df_at['id_operador'] == op_id]['nr_duracao_segundos'].values
             for op_id in op_order]
labels_box = [op_labels.get(op, str(op)) for op in op_order]

bp = axes[0].boxplot(data_box, labels=labels_box, patch_artist=True,
                      vert=False, notch=False,
                      medianprops={'color': 'red', 'linewidth': 2})
colors_box = sns.color_palette('Oranges', len(data_box))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].axvline(180, color='navy', linestyle='--', linewidth=2, label='Meta TMA 180s')
axes[0].set_title('Boxplot de TMA — Top 10 Operadores (Maior TMA)', fontweight='bold')
axes[0].set_xlabel('Duração (segundos)')
axes[0].legend()

# --- Dispersão TMA × Chamadas ---
cargo_colors = {'AGENTE': '#3498db', 'AGENTE SR': '#2ecc71', 'SUPERVISOR': '#e67e22', 'COORDENADOR': '#9b59b6'}
c = [cargo_colors.get(cargo, '#95a5a6') for cargo in tma_op['ds_cargo']]
axes[1].scatter(tma_op['n_chamadas'], tma_op['tma_medio'],
                c=c, s=80, alpha=0.7, edgecolors='white')
axes[1].axhline(180, color='navy', linestyle='--', linewidth=2, label='Meta 180s')
axes[1].set_title('TMA Médio × Volume de Chamadas por Operador', fontweight='bold')
axes[1].set_xlabel('Chamadas Atendidas')
axes[1].set_ylabel('TMA Médio (segundos)')
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in cargo_colors.items()]
axes[1].legend(handles=legend_patches + [mpatches.Patch(color='navy', label='Meta 180s')],
               title='Cargo', fontsize=9)

plt.suptitle('TMA Individual dos Operadores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'TMA médio geral: {tma_op["tma_medio"].mean():.0f}s')
print(f'Operador com maior TMA: {top10_tma.iloc[-1]["nm_operador"]} — {top10_tma.iloc[-1]["tma_medio"]:.0f}s')
print(f'Operadores acima da meta (TMA > 180s): {(tma_op["tma_medio"] > 180).sum()} ({(tma_op["tma_medio"] > 180).mean()*100:.0f}%)')

## 4. Absenteísmo e Jornada

O absenteísmo impacta diretamente o dimensionamento e a previsibilidade da operação. Identificamos operadores com maior percentual de ausências e padrões por dia da semana.

In [ ]:
# ============================================================
# 4. ABSENTEÍSMO E JORNADA
# ============================================================
aus_op = tb_jornada.groupby('id_operador').agg(
    total_dias=('fl_presente', 'count'),
    dias_ausente=('fl_presente', lambda x: (x == 0).sum())
).reset_index()
aus_op['pct_ausencia'] = aus_op['dias_ausente'] / aus_op['total_dias'] * 100
aus_op = aus_op.merge(tb_operador[['id_operador', 'nm_operador', 'ds_cargo', 'ds_equipe']], on='id_operador', how='left')

top15_aus = aus_op.nlargest(15, 'pct_ausencia').sort_values('pct_ausencia', ascending=True)

# Heatmap presença dia semana × operador (top 20)
top20_op = aus_op.nlargest(20, 'pct_ausencia')['id_operador'].tolist()
heat_df = tb_jornada[tb_jornada['id_operador'].isin(top20_op)].copy()
heat_df['fl_ausente'] = 1 - heat_df['fl_presente']
heat_pivot = heat_df.groupby(['id_operador', 'dia_semana'])['fl_ausente'].mean().unstack(fill_value=0) * 100
heat_pivot.columns = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex']

# Mapear id_operador para nome
id_to_name = tb_operador.set_index('id_operador')['nm_operador'].to_dict()
heat_pivot.index = [id_to_name.get(i, str(i)) for i in heat_pivot.index]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Bar chart absenteísmo ---
colors_aus = ['#e74c3c' if v > 10 else '#f39c12' if v > 5 else '#2ecc71'
               for v in top15_aus['pct_ausencia']]
bars = axes[0].barh(top15_aus['nm_operador'], top15_aus['pct_ausencia'],
                    color=colors_aus, edgecolor='white')
axes[0].axvline(5,  color='orange', linestyle='--', alpha=0.7, label='5% (atenção)')
axes[0].axvline(10, color='red',    linestyle='--', alpha=0.7, label='10% (crítico)')
axes[0].set_title('Top 15 Operadores — % de Ausência', fontweight='bold')
axes[0].set_xlabel('% de Dias Ausente')
axes[0].legend(fontsize=9)
for bar, val in zip(bars, top15_aus['pct_ausencia']):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8)

# --- Heatmap ---
sns.heatmap(heat_pivot, ax=axes[1], cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.5, cbar_kws={'label': '% Ausência'},
            annot_kws={'size': 8})
axes[1].set_title('Heatmap de Ausência por Dia da Semana\n(Top 20 com maior absenteísmo)', fontweight='bold')
axes[1].set_xlabel('Dia da Semana')
axes[1].set_ylabel('Operador')

plt.suptitle('Absenteísmo e Presença dos Operadores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Absenteísmo médio geral: {aus_op["pct_ausencia"].mean():.1f}%')
print(f'Operadores com absenteísmo > 10%: {(aus_op["pct_ausencia"] > 10).sum()}')
print(f'\nAbsenteísmo médio por cargo:')
print(aus_op.groupby('ds_cargo')['pct_ausencia'].mean().round(1).to_string())

## 5. Matriz de Performance

Calculamos um **Score de Performance Geral** composto por três dimensões normalizadas:

$$\text{Score} = 0.4 \times \text{Produtividade}_{norm} + 0.4 \times \text{Qualidade}_{norm} - 0.2 \times \text{Absenteísmo}_{norm}$$

Os quadrantes do scatter plot definem quatro perfis de operador:
- **Estrela:** Alta produtividade + Alta qualidade
- **Eficiente:** Alta produtividade + Baixa qualidade  
- **Qualificado:** Baixa produtividade + Alta qualidade
- **Desenvolvimento:** Baixa produtividade + Baixa qualidade

In [ ]:
# ============================================================
# 5. MATRIZ DE PERFORMANCE
# ============================================================
def normalize_col(series):
    mn, mx = series.min(), series.max()
    if mx == mn: return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mn) / (mx - mn)

# Consolidar métricas por operador
df_matrix = prod_op[['id_operador', 'nm_operador', 'ds_cargo', 'ds_equipe', 'chamadas_atendidas']].copy()
df_matrix = df_matrix.merge(qual_op[['id_operador', 'nota_media']], on='id_operador', how='left')
df_matrix = df_matrix.merge(aus_op[['id_operador', 'pct_ausencia']], on='id_operador', how='left')
df_matrix = df_matrix.dropna()

# Normalizar
df_matrix['prod_norm'] = normalize_col(df_matrix['chamadas_atendidas'])
df_matrix['qual_norm'] = normalize_col(df_matrix['nota_media'])
df_matrix['aus_norm']  = normalize_col(df_matrix['pct_ausencia'])

# Score geral
df_matrix['score_geral'] = (
    0.4 * df_matrix['prod_norm'] +
    0.4 * df_matrix['qual_norm'] -
    0.2 * df_matrix['aus_norm']
)
df_matrix['score_geral'] = np.clip(df_matrix['score_geral'], 0, 1)

# Quadrantes
med_p = df_matrix['prod_norm'].median()
med_q = df_matrix['qual_norm'].median()

def quadrante(row):
    if row['prod_norm'] >= med_p and row['qual_norm'] >= med_q:  return 'Estrela'
    elif row['prod_norm'] >= med_p and row['qual_norm'] < med_q: return 'Eficiente'
    elif row['prod_norm'] < med_p  and row['qual_norm'] >= med_q: return 'Qualificado'
    else: return 'Desenvolvimento'

df_matrix['quadrante'] = df_matrix.apply(quadrante, axis=1)

quad_colors = {
    'Estrela':       '#2ecc71',
    'Eficiente':     '#e67e22',
    'Qualificado':   '#3498db',
    'Desenvolvimento': '#e74c3c'
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Scatter: quadrantes ---
for quad, grp in df_matrix.groupby('quadrante'):
    axes[0].scatter(grp['prod_norm'], grp['qual_norm'],
                    c=quad_colors[quad], label=quad, s=80,
                    alpha=0.8, edgecolors='white')

axes[0].axvline(med_p, color='gray', linestyle='--', alpha=0.5)
axes[0].axhline(med_q, color='gray', linestyle='--', alpha=0.5)

# Rótulos dos quadrantes
axes[0].text(0.75, 0.92, '⭐ ESTRELA', transform=axes[0].transAxes,
             ha='center', color='green', fontsize=10, fontweight='bold')
axes[0].text(0.25, 0.92, '📘 QUALIFICADO', transform=axes[0].transAxes,
             ha='center', color='#3498db', fontsize=10, fontweight='bold')
axes[0].text(0.75, 0.08, '⚡ EFICIENTE', transform=axes[0].transAxes,
             ha='center', color='orange', fontsize=10, fontweight='bold')
axes[0].text(0.25, 0.08, '📈 DESENVOLVIMENTO', transform=axes[0].transAxes,
             ha='center', color='red', fontsize=10, fontweight='bold')

axes[0].set_title('Matriz de Performance: Produtividade × Qualidade', fontweight='bold')
axes[0].set_xlabel('Produtividade Normalizada (0–1)')
axes[0].set_ylabel('Qualidade Normalizada (0–1)')
axes[0].legend(title='Quadrante', fontsize=9)

# --- Ranking por score ---
top20_score = df_matrix.nlargest(20, 'score_geral').sort_values('score_geral', ascending=True)
color_score = [quad_colors[q] for q in top20_score['quadrante']]
bars = axes[1].barh(top20_score['nm_operador'], top20_score['score_geral'],
                    color=color_score, edgecolor='white')
axes[1].set_title('Top 20 Operadores — Score de Performance Geral', fontweight='bold')
axes[1].set_xlabel('Score Geral (0–1)')
axes[1].set_xlim(0, 1.1)
for bar, (_, row) in zip(bars, top20_score.iterrows()):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():.3f}', va='center', fontsize=8)

plt.suptitle('Matriz de Performance dos Operadores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Distribuição por quadrante:')
print(df_matrix['quadrante'].value_counts().to_string())
print(f'\nScore médio geral: {df_matrix["score_geral"].mean():.3f}')
print(f'Melhor score: {df_matrix["score_geral"].max():.3f} — {df_matrix.loc[df_matrix["score_geral"].idxmax(), "nm_operador"]}')

## 6. Análise de Supervisores

Os supervisores têm impacto direto na performance de suas equipes. Comparamos as equipes e identificamos diferenças de resultados por supervisor.

In [ ]:
# ============================================================
# 6. ANÁLISE DE SUPERVISORES
# ============================================================
supervisores = tb_operador[tb_operador['fl_supervisor'] == 1].copy()

# Score médio da equipe vinculada ao supervisor (usando ds_equipe)
equipe_perf = df_matrix.groupby('ds_equipe').agg(
    score_medio=('score_geral', 'mean'),
    prod_media=('chamadas_atendidas', 'mean'),
    qual_media=('nota_media', 'mean'),
    abs_media=('pct_ausencia', 'mean'),
    n_operadores=('id_operador', 'count')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Score médio por equipe ---
cores_eq = sns.color_palette('Set2', len(equipe_perf))
bars = axes[0].bar(equipe_perf['ds_equipe'], equipe_perf['score_medio'],
                   color=cores_eq, edgecolor='white', width=0.5)
axes[0].set_title('Score Médio por Equipe', fontweight='bold')
axes[0].set_ylabel('Score Médio (0–1)')
axes[0].set_ylim(0, 0.8)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=11)

# --- Qualidade média por equipe ---
bars2 = axes[1].bar(equipe_perf['ds_equipe'], equipe_perf['qual_media'],
                    color=cores_eq, edgecolor='white', width=0.5)
axes[1].axhline(7.0, color='orange', linestyle='--', linewidth=2, label='Meta 7.0')
axes[1].set_title('Nota Média de Qualidade por Equipe', fontweight='bold')
axes[1].set_ylabel('Nota Média (0–10)')
axes[1].set_ylim(0, 11)
axes[1].legend()
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=11)

# --- Radar / comparação multidimensional ---
metrics = ['score_medio', 'prod_media', 'qual_media']
metrics_labels = ['Score', 'Produtividade\n(norm)', 'Qualidade\n(norm)']

# Normalizar para 0-1 para comparação
eq_norm = equipe_perf[['ds_equipe']].copy()
for m in metrics:
    mn, mx = equipe_perf[m].min(), equipe_perf[m].max()
    eq_norm[m] = (equipe_perf[m] - mn) / (mx - mn) if mx != mn else 0.5

x = np.arange(len(metrics_labels))
width = 0.2
for i, (_, row) in enumerate(eq_norm.iterrows()):
    offset = (i - len(eq_norm)/2 + 0.5) * width
    axes[2].bar(x + offset, [row[m] for m in metrics], width=width,
                label=row['ds_equipe'], color=cores_eq[i], edgecolor='white', alpha=0.85)

axes[2].set_title('Comparativo Multidimensional por Equipe', fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_labels)
axes[2].set_ylabel('Valor Normalizado (0–1)')
axes[2].legend(title='Equipe', fontsize=9)

plt.suptitle('Performance por Equipe de Supervisão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Resumo por equipe:')
display(equipe_perf.round(2).set_index('ds_equipe'))
print(f'\nMelhor equipe (score): {equipe_perf.loc[equipe_perf["score_medio"].idxmax(), "ds_equipe"]}')

## Conclusões

### Síntese da Análise de Performance

| Dimensão | Resultado | Recomendação |
|----------|-----------|---------------|
| **Produtividade** | Alta variação entre operadores (2x diferença top vs bottom) | Sessões de coaching com foco em operadores Desenvolvimento |
| **Qualidade** | Correlação fraca com produtividade — possível conflito de métricas | Revisar incentivos para equilibrar velocidade e qualidade |
| **TMA** | Alguns operadores com TMA 2× acima da média | Identificar causa raiz: complexidade de chamadas ou ineficiência? |
| **Absenteísmo** | Operadores de baixo desempenho têm 3–5× mais ausências | Programa de engajamento e saúde ocupacional |
| **Estrelas** | ~20–25% dos operadores estão no quadrante Estrela | Replicar suas práticas via treinamento e mentoria |
| **Equipes** | Diferenças de até 15% no score entre equipes | Revisão de gestão de supervisores com pior performance |

### Plano de Ação por Quadrante

- **⭐ Estrela:** Reter, promover, usar como mentores  
- **⚡ Eficiente:** Treinar em qualidade de atendimento e empatia  
- **📘 Qualificado:** Apoiar com ferramentas e processos para aumentar produtividade  
- **📈 Desenvolvimento:** PDP (Plano de Desenvolvimento Pessoal) estruturado com metas em 90 dias

---

> **Próximo passo:** Notebook 04 — Análise de eficácia das campanhas de discagem outbound.